# 06 — SHAP Explainability (Phase 7)

Wraps `src/explainability/` for interactive inspection of global and local
explanations. SHAP is always computed from the **uncalibrated** `raw_pipeline`'s
underlying tree model — never from the `CalibratedClassifierCV` wrapper.

**Not executed in this environment** — outputs are empty until run locally.

In [ ]:
from src.config import get_default_config
from src.data.load import load_and_validate_data
from src.data.split import train_test_split_data
from src.models.pipeline import build_model_pipeline
from src.calibration.calibrator import fit_calibrated_model_with_split
from src.explainability.shap_explainer import transform_for_shap, build_feature_name_mapping
from src.explainability.global_explanation import generate_global_shap_report
from src.explainability.local_explanation import explain_prediction

config = get_default_config()
config.data.target_column = "<set explicitly — dataset-specific>"
config.validate()

In [ ]:
df, schema = load_and_validate_data(config.data)
X_train, X_test, y_train, y_test = train_test_split_data(
    df, config.data.target_column, config.split
)

base_pipeline = build_model_pipeline(config.model, config.preprocessing)
fit_result, X_calib, y_calib = fit_calibrated_model_with_split(
    base_pipeline, X_train, y_train, config.calibration
)

raw_pipeline = fit_result.raw_pipeline
calibrated_model = fit_result.calibrated_model

## Global explanation

Writes `shap_summary.png`, `shap_bar.png`, `feature_importance.csv` under
`config.artifacts.shap_reports_dir` and returns a summary dict.

In [ ]:
global_summary = generate_global_shap_report(
    raw_pipeline=raw_pipeline,
    X_sample=X_train,
    numerical_features=config.preprocessing.numerical_features,
    categorical_features=config.preprocessing.categorical_features,
    output_dir=f"{config.artifacts.shap_reports_dir}/global",
    max_display=config.explainability.max_display_features,
    sample_size=config.explainability.global_sample_size,
    random_state=config.explainability.random_state,
)
global_summary

## Local explanation (single applicant)

Uses the calibrated model for the probability/decision, and the raw pipeline's
tree model for SHAP attribution — these are deliberately different objects.

In [ ]:
example_row = X_test.iloc[[0]]

explanation = explain_prediction(
    raw_pipeline=raw_pipeline,
    calibrated_model=calibrated_model,
    threshold=0.5,  # replace with the artifact-persisted threshold from Phase 5/8
    input_df=example_row,
    feature_schema=schema,
    numerical_features=config.preprocessing.numerical_features,
    categorical_features=config.preprocessing.categorical_features,
    top_n=config.explainability.top_n_local_features,
)
explanation

## Reminders

- SHAP values here are in log-odds/margin space — they are **not** an exact
  decomposition of `calibrated_model.predict_proba`.
- SHAP attributions are associative, not causal. `explanation['explanation_note']`
  carries this disclaimer through to any downstream (future API) consumer.
- This output is decision support only, not an automated final lending decision.